# Questão 8 — Sistema de Recomendação

## Objetivo

Construir um motor de recomendação simples baseado em similaridade de comportamento de compra entre produtos.

O item de referência será:

**GPS Garmin Vortex Maré Drift**

## Abordagem

A recomendação será feita a partir de uma matriz de interação usuário × produto, onde:

- cada linha representa um cliente
- cada coluna representa um produto
- o valor será 1 quando o cliente tiver comprado o produto ao menos uma vez
- o valor será 0 caso contrário

Com essa matriz, será calculada a **similaridade de cosseno entre produtos**, permitindo identificar quais itens possuem padrão de compra mais semelhante entre os clientes.

## Fonte de dados

Serão utilizadas as tabelas analíticas da camada gold:

- `lh_nautical.04_gold.fct_vendas`
- `lh_nautical.04_gold.dim_produto`

In [0]:
# Bibliotecas utilizadas na construção do recomendador
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [0]:
# Leitura das tabelas
df_vendas = spark.table("lh_nautical.04_gold.fct_vendas")
df_produtos = spark.table("lh_nautical.04_gold.dim_produto")

# Conversão para pandas para facilitar a manipulação da matriz usuário-item
pdf_vendas = df_vendas.toPandas()
pdf_produtos = df_produtos.toPandas()

# Conferência inicial
print("Linhas fct_vendas:", len(pdf_vendas))
print("Linhas dim_produto:", len(pdf_produtos))

## Preparação da base

Como a fato de vendas contém os identificadores de cliente e produto, e a dimensão de produto contém o nome do item, será feita uma junção entre as duas tabelas para permitir a identificação do produto de referência dentro da matriz de similaridade.

In [0]:
# Junção entre fato de vendas e dimensão de produto
# Necessária para obter o nome do produto de referência
df_base = pdf_vendas.merge(
    pdf_produtos[["product_id", "product_name"]],
    on="product_id",
    how="left"
)

# Visualização inicial da base analítica
df_base.head()

In [0]:
# Criação de indicador binário de compra
# Independentemente da quantidade comprada, basta saber se o cliente comprou o produto ao menos uma vez
df_base["comprou"] = 1

# Construção da matriz usuário × produto
# Linhas: customer_id
# Colunas: product_id
# Valores: 1 se comprou ao menos uma vez, 0 caso contrário
matriz_usuario_item = pd.pivot_table(
    df_base,
    index="customer_id",
    columns="product_id",
    values="comprou",
    aggfunc="max",
    fill_value=0
)

# Garantir tipo inteiro para facilitar leitura
matriz_usuario_item = matriz_usuario_item.astype(int)

matriz_usuario_item.head()

## Cálculo da similaridade entre produtos

A matriz construída está no formato usuário × produto.  
Como a similaridade deve ser calculada entre produtos, será utilizada a matriz transposta, onde cada produto passa a ser representado por um vetor de clientes.

In [0]:
# Transposição da matriz para o formato produto × usuário
# Cada linha passa a representar um produto
matriz_produto_usuario = matriz_usuario_item.T

matriz_produto_usuario.head()

In [0]:
# Cálculo da similaridade de cosseno entre os vetores dos produtos
similaridade = cosine_similarity(matriz_produto_usuario)

# Transformação do resultado em DataFrame para facilitar consulta
similaridade_df = pd.DataFrame(
    similaridade,
    index=matriz_produto_usuario.index,
    columns=matriz_produto_usuario.index
)

similaridade_df.head()

## Identificação do produto de referência

Antes de gerar o ranking, será identificado o `product_id` correspondente ao item **GPS Garmin Vortex Maré Drift**.

In [0]:
# Definição do produto de referência
produto_referencia = "GPS Garmin Vortex Maré Drift"

# Busca do product_id correspondente
produto_ref_row = pdf_produtos[
    pdf_produtos["product_name"] == produto_referencia
][["product_id", "product_name"]].drop_duplicates()

produto_ref_row

In [0]:
# Extração do product_id do produto de referência
produto_ref_id = int(produto_ref_row["product_id"].iloc[0])

print("Produto de referência:", produto_referencia)
print("ID do produto de referência:", produto_ref_id)

In [0]:
# Série com a similaridade do produto de referência em relação a todos os demais
ranking_similaridade = (
    similaridade_df[produto_ref_id]
    .reset_index()
)

# Ajuste de nomes das colunas
ranking_similaridade.columns = ["product_id", "similaridade"]

# Remoção do próprio produto de referência do ranking
ranking_similaridade = ranking_similaridade[
    ranking_similaridade["product_id"] != produto_ref_id
].copy()

# Ordenação decrescente da similaridade
ranking_similaridade = ranking_similaridade.sort_values(
    by="similaridade",
    ascending=False
)

In [0]:
# Inclusão do nome do produto para facilitar interpretação do ranking
ranking_similaridade = ranking_similaridade.merge(
    pdf_produtos[["product_id", "product_name"]],
    on="product_id",
    how="left"
)

# Ranking final dos 5 produtos mais similares
top5_similares = ranking_similaridade.head(5)

top5_similares

## Ranking dos produtos mais similares

A tabela abaixo apresenta os 5 produtos com maior similaridade de compra em relação ao item de referência, desconsiderando o próprio GPS.

In [0]:
# Exibição final do ranking com foco analítico
top5_similares[["product_id", "product_name", "similaridade"]]

In [0]:
# Produto com maior similaridade ao item de referência
produto_mais_similar = top5_similares.iloc[0]

print("id_produto com maior similaridade:", int(produto_mais_similar["product_id"]))
print("produto recomendado:", produto_mais_similar["product_name"])
print("similaridade:", round(produto_mais_similar["similaridade"], 4))

## Questão 8.2 — Validação

O `id_produto` com maior similaridade ao **GPS Garmin Vortex Maré Drift** foi **94**.

## Questão 8.3 — Explicação

### Como a matriz foi construída?

A matriz foi construída no formato usuário × produto, com `customer_id` nas linhas e `product_id` nas colunas. O valor de cada célula foi definido como 1 quando o cliente comprou aquele produto ao menos uma vez e 0 quando nunca comprou. A quantidade comprada foi desconsiderada, conforme exigido pela questão.


### O que significa a similaridade de cosseno nesse contexto?

A similaridade de cosseno mede o quão parecidos são dois produtos com base no padrão de clientes que os compraram. Se dois produtos forem comprados com frequência pelos mesmos clientes, seus vetores terão direção semelhante e a similaridade será mais alta. Nesse contexto, isso indica afinidade de comportamento de compra entre itens.

### Uma limitação desse método de recomendação

Uma limitação importante é que o método considera apenas coocorrência de compra e não incorpora contexto adicional, como ordem temporal, quantidade comprada, categoria do produto, preço ou perfil do cliente. Além disso, produtos com poucas compras podem gerar similaridades pouco estáveis.